# This notebook converts the .mat data to netcdf format, 
# add the necessary metadata and plot the data

## load library

In [45]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os, sys
import json
# Import your own modules
%reload_ext autoreload
%autoreload 2

# from src.netcdf import mat_to_xarray
sys.path.append('/Users/yugao/UOP/ORS-processing/src')
from metadata import create_json
from netcdf import read_mat_file, read_metadata_json, save_json, mat_to_xarray, save_to_netcdf


## Depth parameters from mooring diagram and mooring log 

In [50]:
depth_parameters = {
    'water_depth_from_mooring_diagram': 4450,  # consult mooring diagram
    'water_depth_from_ship_uncorrected': 4562.2,      # uncorrected water depth 
    'water_depth_from_ship_corrected': 4538.97,        # Replace with actual value, best water depth
    'instrument_depth_from_mooring_diagram': 4414,  # diagram depth (4450 m) - height above bottom (36 m) = 
    'instrument_depth_from_mooring_log': 4532.97,      # corrected depth (4538.97 m) - height above bottom (36 m) = 
    'instrument_height_above_bottom': 36       
    #  Anchor (1) + chain (5) + nystron (20) + chain (5) + releases (1) + chain (5) 
    # + 6 terminations at 0.25 ea (1.5) 
    # + distance from termination on SBE cage to sensor (0.5) = 39 m
}

## read mat file

In [51]:
mat_filepath = '../../data/external/stratus12_sbe16_1876.mat'
mat_data = read_mat_file(mat_filepath)
# Inspect the structure of mat_data
print(mat_data.keys()) 

dict_keys(['__header__', '__version__', '__globals__', 'meta', 'latitude', 'longitude', 'depth', 'mday', 'temp', 'cond', 'sal', 'sal_sbe', 'year', 'yday'])


## convert mat file to netcdf

In [52]:
sbe16_metadata = mat_data['meta']

## Meta data: we will add depth paramter for deep T/S

In [53]:
ds = xr.open_dataset('/Users/yugao/UOP/ORS-processing/data/external/OS_NTAS_2016_D_TS.nc')

In [54]:
original_json_path = '../../data/metadata/OS_NTAS_2016_D_TS.json'
new_json_path = '../../data/metadata/stratus_OS_NTAS_2016_D_TS.json'

# Update metadata dictionary with depth parameters
metadata = {
    'dimensions': dict(ds.sizes),  # Updated based on FutureWarning advice
    'coordinates': {name: {'data': coord.values.tolist(), 'attrs': dict(coord.attrs)} for name, coord in ds.coords.items()},
    'variables': {name: {'dims': var.dims, 'data_type': str(var.dtype), 'attrs': dict(var.attrs)} for name, var in ds.data_vars.items()},
    'attributes': dict(ds.attrs),
    'depth_parameters': depth_parameters  # Adding depth parameters to metadata
}

# Export updated metadata to a JSON file
with open('../../data/processed/OS_NTAS_2016_D_TS_with_depth_paramter.json', 'w') as f_json:
    json.dump(metadata, f_json, indent=4)
    
save_json(original_json_path, new_json_path)

In [55]:
metadata

{'dimensions': {'TIME': 8726,
  'DEPTH': 7,
  'DEPTH_sal': 6,
  'LATITUDE': 1,
  'LONGITUDE': 1},
 'coordinates': {'TIME': {'data': [1454448600000003328,
    1454452200000000000,
    1454455799999996672,
    1454459400000003328,
    1454463000000000000,
    1454466599999996672,
    1454470200000003328,
    1454473800000000000,
    1454477399999996672,
    1454481000000003328,
    1454484600000000000,
    1454488199999996672,
    1454491800000003328,
    1454495400000000000,
    1454498999999996672,
    1454502600000003328,
    1454506200000000000,
    1454509799999996672,
    1454513400000003328,
    1454517000000000000,
    1454520599999996672,
    1454524200000003328,
    1454527800000000000,
    1454531399999996672,
    1454535000000003328,
    1454538600000000000,
    1454542199999996672,
    1454545800000003328,
    1454549400000000000,
    1454552999999996672,
    1454556600000003328,
    1454560200000000000,
    1454563799999996672,
    1454567400000003328,
    14545710000000000

### Function testing: avoid duplicates!!!

In [56]:
def map_matlab_metadata_to_json(mat_data):
    # Start with an empty dictionary that will be populated with mapped values
    json_structure = {
        'dimensions': {},  # Updated based on FutureWarning advice
        'coordinates': {},
        'variables': {},
        'attributes': {},
        'depth_parameters': depth_parameters  # Adding depth parameters to metadata
    }

    # Dimensions

    dimensions = {
        'TIME': len(mat_data['TIME']) if 'TIME' in mat_data else 0,
        'LATITUDE': 1,  # Since 'latitude' is a single float value
        'LONGITUDE': 1,  # Since 'longitude' is a single float value
        'DEPTH': len(mat_data['DEPTH']) if 'DEPTH' in mat_data else 0,
    }

    coordinates = {
    'TIME': {
        'data': mat_data['TIME'].tolist() if 'TIME' in mat_data else [],
        # Add attributes as needed
    },
    'LATITUDE': {
        'data': [mat_data['latitude']] if 'latitude' in mat_data else [],
        # Add attributes as needed
    },
    'LONGITUDE': {
        'data': [mat_data['longitude']] if 'longitude' in mat_data else [],
        # Add attributes as needed
    },
    'DEPTH': {
        'data': mat_data['DEPTH'].tolist() if 'DEPTH' in mat_data else [],
        # Add attributes as needed
    },
    }

    # Variable

    variables = {
    'TEMP': {
        'dims': ('TIME', 'DEPTH'),
        'data_type': 'float64',
        'attrs': {
            'long_name': 'Sea Water Temperature',
            'units': 'degree_Celsius',
            'QC_indicator': 'Good data',
            # Additional variable-specific metadata...
        }
    },
    'CNDC': {
        'dims': ('TIME', 'DEPTH_sal'),
        'data_type': 'float64',
        'attrs': {
            'long_name': 'Sea Water Electrical Conductivity',
            'units': 'S m-1',
            # Additional variable-specific metadata...
        }
    },
    'PSAL': {
        'dims': ('TIME', 'DEPTH_sal'),
        'data_type': 'float64',
        'attrs': {
            'long_name': 'Sea Water Practical Salinity',
            'units': '1',
            # Additional variable-specific metadata...
        }
    },
    # Include other variables as needed...
    }




    # Manually extract and map values from `mat_data` to `json_structure`
    # Example: Mapping a simple value
    # json_structure['metadata']['site_code'] = mat_data['site'][0]  # Assuming 'site' is a key in `mat_data`
    json_structure['coordinates']['latitude'] = mat_data['latitude']
    json_structure['coordinates']['longitude'] = mat_data['longitude']
    json_structure['coordinates']['depth'] = mat_data['depth']

    # Mapping Variables: Mapping variables like temperature, conductivity, salinity, etc., 
    # requires converting NumPy arrays to lists and adding any available attributes like units.
    json_structure['variables']['temp'] = {
        'data': mat_data['temp'].tolist(),  # Convert array to list
        'attributes': {
            'units': 'degrees Celsius'  # Example; adjust as necessary
        }
    }
    # Repeat for 'cond', 'sal', 'sal_sbe' with appropriate attributes

    
    return json_structure


In [57]:
len(mat_data['longitude'])

TypeError: object of type 'float' has no len()

In [58]:
sbe16_stratus12 = map_matlab_metadata_to_json(mat_data)
sbe16_stratus12

{'dimensions': {},
 'coordinates': {'latitude': -19.93844,
  'longitude': -85.29266333333334,
  'depth': 4502},
 'variables': {'temp': {'data': [17.313,
    17.214,
    17.1008,
    16.9734,
    16.8357,
    16.6908,
    16.5363,
    16.3729,
    16.2077,
    16.0421,
    15.8754,
    15.709,
    15.5451,
    15.3836,
    15.2283,
    15.0799,
    14.934,
    14.7908,
    14.6443,
    14.4967,
    14.3475,
    14.2029,
    14.0625,
    13.9344,
    13.8363,
    13.7772,
    13.7572,
    13.7749,
    13.8318,
    13.9211,
    14.1155,
    14.6669,
    15.2854,
    15.7708,
    16.1699,
    16.5377,
    16.8605,
    17.1487,
    17.3684,
    17.5413,
    17.6817,
    17.7786,
    17.8113,
    17.7779,
    17.7208,
    17.6258,
    17.4902,
    17.3221,
    17.1255,
    16.8965,
    16.6462,
    16.3843,
    16.1178,
    15.8539,
    15.612,
    15.406,
    15.242,
    15.0995,
    14.9696,
    14.8577,
    14.7602,
    14.6749,
    14.6012,
    14.5369,
    14.4851,
    14.4434,
    14.4

### This is an example of all the keys we want for our dataset from NTAS

In [15]:
## This is an existing meta data for reference
## we should update
metadata_filepath = '../../data/metadata/OS_NTAS_2016_D_TS.json'
metadata = read_metadata_json(metadata_filepath)
# Inspect metadata contents
metadata.keys()

dict_keys(['dimensions', 'coordinates', 'variables', 'attributes', 'depth_parameters'])

In [30]:
mat_data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: MACI64, Created on: Mon Mar 10 09:38:17 2014',
 '__version__': '1.0',
 '__globals__': [],
 'meta': array(('Stratus', '12', 'Stratus Ocean Reference Station', 'Robert Weller', array(('surface mooring', 2012, '27-May-2012 22:03:04', '03-Mar-2014 12:54:00', 735016.9791666666, 735624.5833333334, array([735016.9187963 , 735666.45549769]), 607.6645370370243, 644.6187037036289, 4539, 65, 3.7, 'Melville MV12-07', 'Ron Brown RB14-01', array((6.97132, -6.79155, 'NOAA > NESDIS > NGDC', 'http://www.ngdc.noaa.gov/geomag-web/#declination', 'IGRF11', '27-Mar-2013'),
       dtype=[('value_to_be_applied', 'O'), ('changing_by_MinutesPerYear', 'O'), ('source', 'O'), ('URL', 'O'), ('model', 'O'), ('midpoint_date', 'O')])),
       dtype=[('type', 'O'), ('start_year', 'O'), ('anchor_over_time', 'O'), ('buoy_recovery_time', 'O'), ('data_start', 'O'), ('adrift_time', 'O'), ('anchor_times', 'O'), ('days_on_station', 'O'), ('duration', 'O'), ('water_depth_m', 'O')

In [16]:
sbe16_metadata.keys()

AttributeError: 'numpy.ndarray' object has no attribute 'keys'

In [9]:
# mapped_json = map_matlab_metadata_to_json(mat_data)
# save_json(mapped_json, '/path/to/your/output_file.json')

NameError: name 'map_matlab_metadata_to_json' is not defined

In [ ]:
# sbe16_metadata.tolist()

In [ ]:
# Now call the mat_to_xarray() function
ds = mat_to_xarray(mat_data)

In [ ]:
mat_data['mday']

### plot temperature 

In [ ]:
plt.plot(mat_data['mday'][10:], mat_data['temp'][:10])

In [ ]:
plt.plot(mat_data['mday'][:1000], mat_data['temp'][:1000])

In [ ]:
plt.plot(mat_data['mday'][1000:10000], mat_data['temp'][1000:10000])

In [ ]:
ds = mat_to_xarray(mat_data)
ds
# Verify that the DataSet is created correctly
# Check ds's content, dimensions, and attributes based on your requirements

In [ ]:
output_filepath = '../data/sample_output.nc'
# Replace with your actual file path